# Idealista Web Scraper — Local, No-AWS Guided Walkthrough

**FEATURE-002, task 1.11.** This notebook is a guided, cell-by-cell surface for developing and
testing the Idealista web-scraper OOP core (`data_collection.scraper`) **entirely locally**: no AWS
credentials, no paid proxy vendor. It wires `NullProxyProvider` + `LocalListingRepository` so every
cell can be run by anyone who clones this repo.

**Compliance / ToS note (REVIEW-FEATURE-002 finding H2):** this scraper targets Idealista's public
search-results pages to supplement the official API collector (capped at 100 listings/month). Please
scrape responsibly:
- keep the randomised inter-page delay (`ScrapeOrchestrator`'s default 2.0–4.5s) — never tighten it,
- respect the `SCRAPER_ENABLED` kill switch (`config.py`) — set it to `false` to disable any run,
- prefer the official API collector (`bronze_collector.py`) as the primary source where its quota
  suffices; this scraper is a supplement, not a replacement,
- this notebook makes **at most a handful of requests** per run (1 page for the guided fetch, 2–3
  pages per operation for the smoke-test run below) — it is not intended to scrape the full inventory
  interactively.


In [1]:
import sys
from pathlib import Path

# Robustly locate src/etl regardless of whether this notebook is run from
# the repo root or from src/notebooks/ (Jupyter's default cwd).
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
_etl_root = None
for _base in _candidates:
    for _rel in ("src/etl", "etl"):
        _candidate = _base / _rel
        if (_candidate / "data_collection" / "scraper").exists():
            _etl_root = _candidate
            break
    if _etl_root is not None:
        break

if _etl_root is None:
    raise RuntimeError(
        "Could not locate src/etl/data_collection/scraper; run this notebook "
        "from the repository root or from src/notebooks/."
    )

sys.path.insert(0, str(_etl_root))
print(f"Using etl root: {_etl_root}")


Using etl root: /Users/leopoldwalther/Documents/projects/vlc-real-estate/vlc-real-estate-analytics/src/etl


## 1. Imports

Import the domain model, strategies, URL builder, proxy/fetcher/parser/repository adapters, and the
orchestrator — the same public surface the CLI (`python -m data_collection.scraper`) uses.


In [2]:
import json
from datetime import datetime

import pandas as pd

from data_collection.scraper.domain import Listing, ListingCollection
from data_collection.scraper.urls import SaleStrategy, RentStrategy, IdealistaUrlBuilder
from data_collection.scraper.proxies import NullProxyProvider
from data_collection.scraper.fetcher import CloudscraperFetcher
from data_collection.scraper.parser import IdealistaListingParser
from data_collection.scraper.repository import LocalListingRepository
from data_collection.scraper.orchestrator import ScrapeOrchestrator
from data_collection.scraper.errors import ScraperError

print("Imports OK")


Imports OK


## 2. Build the local object graph

`NullProxyProvider` needs no credentials (`get_proxy()` always returns `None`); `LocalListingRepository`
writes JSON pages to `data/s3/` (the same convention as the API collector's local runs), so nothing here
touches AWS.


In [3]:
OUTPUT_DIR = "../../data/s3"  # relative to src/notebooks/; adjust if running from elsewhere

proxy_provider = NullProxyProvider()
fetcher = CloudscraperFetcher(proxy_provider=proxy_provider, max_retries=1, base_delay=0.5)  # low retry budget: keeps this notebook fast when live fetches are blocked
parser = IdealistaListingParser()
repository = LocalListingRepository(output_dir=OUTPUT_DIR)

orchestrator = ScrapeOrchestrator(
    fetcher=fetcher,
    parser=parser,
    repository=repository,
    proxy_provider=proxy_provider,
)

print("Object graph ready: NullProxyProvider + LocalListingRepository (no AWS/proxy creds)")


Object graph ready: NullProxyProvider + LocalListingRepository (no AWS/proxy creds)


## 3. Fetch one page directly

Fetch page 1 of the **sale** search results via the fetcher directly (bypassing the orchestrator) so we
can inspect the raw HTML. Idealista's Cloudflare protection may block `cloudscraper` from a sandboxed /
data-centre IP — this is wrapped in a `try/except` so the notebook keeps running end-to-end even when
live network access isn't available; a human running this later with working network/proxy access can
proceed from the printed HTML snippet.


In [4]:
url_builder = IdealistaUrlBuilder(SaleStrategy())
page_1_url = url_builder.build(page=1)
print(f"Fetching: {page_1_url}")

html = None
try:
    html = fetcher.fetch(page_1_url)
    print(f"Fetched {len(html)} characters")
    print(html[:500])
except ScraperError as exc:
    print(
        "WARNING: live fetch failed in this environment (likely blocked by "
        f"Idealista's anti-bot protection): {exc}\n"
        "This is expected in sandboxed/CI environments without a working "
        "residential proxy. Falling back to the frozen test fixture so the "
        "rest of the notebook can still be demonstrated end-to-end."
    )
    fixture_path = Path("../etl/data_collection/tests/fixtures/search_results_sale.html")
    if fixture_path.exists():
        html = fixture_path.read_text(encoding="utf-8")
        print(f"Loaded {len(html)} characters from the frozen fixture instead.")


Fetch attempt 1/2 for https://www.idealista.com/en/venta-viviendas/valencia-valencia/?pagina=1 failed (status=403, error=None); rotating proxy and retrying in 0.5s


Fetching: https://www.idealista.com/en/venta-viviendas/valencia-valencia/?pagina=1


This is expected in sandboxed/CI environments without a working residential proxy. Falling back to the frozen test fixture so the rest of the notebook can still be demonstrated end-to-end.
Loaded 5963 characters from the frozen fixture instead.


## 4. Parse and inspect as a DataFrame

Parse whatever HTML we obtained above (live or the fixture fallback) into `Listing` objects and view
them as a `pandas.DataFrame`.


In [5]:
listings_df = pd.DataFrame()

if html:
    collection = parser.parse(html, operation="sale")
    listings_df = pd.DataFrame([listing.to_dict() for listing in collection])
    print(f"Parsed {len(collection)} listings")
else:
    print("No HTML available (live fetch failed and no fixture found); skipping parse step.")

listings_df.head()


Parsed 5 listings


,propertyCode,thumbnail,floor,price,propertyType,operation,size,exterior,rooms,bathrooms,address,province,municipality,district,country,neighborhood,latitude,longitude,url
0,107517743,None,2nd,695000.0,None,sale,132.0,None,3,None,"Flat in calle de Ruzafa, Ruzafa",None,None,None,es,None,None,None,https://www.idealista.com/inmueble/107517743/
1,108234501,None,1st,245000.0,None,sale,78.0,None,2,None,Flat in Benimaclet,None,None,None,es,None,None,None,https://www.idealista.com/inmueble/108234501/
2,109887654,None,5th,890000.0,None,sale,168.0,None,4,None,Flat in El Pla del Remei,None,None,None,es,None,None,None,https://www.idealista.com/inmueble/109887654/
3,110456789,None,Ground,310500.0,None,sale,95.0,None,3,None,Flat in Extramurs,None,None,None,es,None,None,None,https://www.idealista.com/inmueble/110456789/
4,111222333,None,3rd,415000.0,None,sale,64.0,None,2,None,"Flat in Ruzafa, near Mercado de Ruzafa",None,None,None,es,None,None,None,https://www.idealista.com/inmueble/111222333/


Inspect the DataFrame's dtypes and non-null counts (useful for spotting which fields the parser left as `None` on this page).

In [6]:
if not listings_df.empty:
    listings_df.info()
else:
    print("listings_df is empty — nothing to inspect.")


<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   propertyCode  5 non-null      str    
 1   thumbnail     0 non-null      object 
 2   floor         5 non-null      str    
 3   price         5 non-null      float64
 4   propertyType  0 non-null      object 
 5   operation     5 non-null      str    
 6   size          5 non-null      float64
 7   exterior      0 non-null      object 
 8   rooms         5 non-null      int64  
 9   bathrooms     0 non-null      object 
 10  address       5 non-null      str    
 11  province      0 non-null      object 
 12  municipality  0 non-null      object 
 13  district      0 non-null      object 
 14  country       5 non-null      str    
 15  neighborhood  0 non-null      object 
 16  latitude      0 non-null      object 
 17  longitude     0 non-null      object 
 18  url           5 non-null      str    
dtypes:

## 5. Full sale + rent run (bounded smoke test)

Run the `ScrapeOrchestrator` for both operations, writing every page to `data/s3/`. **Trade-off:** a
true "full Valencia inventory" run can be dozens of pages per operation; for a reasonable local smoke
test (and to stay polite/bounded while Idealista's anti-bot protection may block requests anyway), this
capped run uses `max_pages=2`. Remove the cap for a real full-inventory run once network/proxy access is
confirmed to work.


In [7]:
results = {}
for strategy in (SaleStrategy(), RentStrategy()):
    label = strategy.operation_label
    try:
        collection = orchestrator.scrape(strategy, max_pages=2)
        results[label] = collection
        print(f"{label}: scraped {len(collection)} unique listings (<= 2 pages)")
    except ScraperError as exc:
        print(
            f"WARNING: {label} scrape failed in this environment (likely blocked by "
            f"Idealista's anti-bot protection): {exc}"
        )
        results[label] = ListingCollection()

results


Fetch attempt 1/2 for https://www.idealista.com/en/venta-viviendas/valencia-valencia/?pagina=1 failed (status=403, error=None); rotating proxy and retrying in 0.5s


Fetch attempt 1/2 for https://www.idealista.com/en/alquiler-viviendas/valencia-valencia/?pagina=1 failed (status=403, error=None); rotating proxy and retrying in 0.5s


{'sale': <data_collection.scraper.domain.ListingCollection at 0x122eddd90>,
 'rent': <data_collection.scraper.domain.ListingCollection at 0x12358bce0>}

## 6. Validate field coverage against an existing API-collected JSON

Load one existing API-collected JSON page (`data/s3/sale_20250413_120045_3.json`) and compare its
`elementList[0]` keys against the scraper's `Listing.to_dict()` keys, so we can see at a glance which
fields the scraper still needs to add (or which API-only fields we deliberately don't replicate).


In [8]:
api_json_path = Path(OUTPUT_DIR) / "sale_20250413_120045_3.json"

if api_json_path.exists():
    api_data = json.loads(api_json_path.read_text(encoding="utf-8"))
    api_keys = set(api_data["elementList"][0].keys())

    # A representative scraper Listing (fields don't matter, only its to_dict() keys).
    sample_listing = Listing(
        property_code="1",
        price=100000,
        size=50,
        rooms=2,
        bathrooms=1,
        address="Sample address",
        url="https://www.idealista.com/inmueble/1/",
        operation="sale",
    )
    scraper_keys = set(sample_listing.to_dict().keys())

    print("Fields present in the API JSON but NOT produced by the scraper:")
    print(sorted(api_keys - scraper_keys))
    print()
    print("Fields produced by the scraper but NOT present in the API JSON:")
    print(sorted(scraper_keys - api_keys))
    print()
    print("Fields present in both:")
    print(sorted(api_keys & scraper_keys))
else:
    print(
        f"Could not find {api_json_path} — skipping the coverage comparison. "
        "Run the API collector at least once locally, or adjust api_json_path "
        "to point at any existing data/s3/*.json file."
    )


Fields present in the API JSON but NOT produced by the scraper:
['change', 'description', 'detailedType', 'distance', 'externalReference', 'has360', 'has3DTour', 'hasLift', 'hasPlan', 'hasStaging', 'hasVideo', 'highlight', 'newDevelopment', 'notes', 'numPhotos', 'parkingSpace', 'priceByArea', 'priceInfo', 'savedAd', 'showAddress', 'status', 'suggestedTexts', 'topNewDevelopment', 'topPlus']

Fields produced by the scraper but NOT present in the API JSON:
[]

Fields present in both:
['address', 'bathrooms', 'country', 'district', 'exterior', 'floor', 'latitude', 'longitude', 'municipality', 'neighborhood', 'operation', 'price', 'propertyCode', 'propertyType', 'province', 'rooms', 'size', 'thumbnail', 'url']


## Summary

- The scraper's OOP core (domain, urls, proxies, fetcher, parser, repository, orchestrator) runs fully
  locally with `NullProxyProvider` + `LocalListingRepository` — no AWS, no proxy credentials.
- Live fetches may be blocked by Idealista's Cloudflare protection from this environment; every
  network-dependent cell above degrades gracefully (prints a warning, keeps the notebook running) rather
  than crashing.
- Field-coverage gaps (if any) between the API JSON and the scraper's `Listing.to_dict()` are printed in
  section 6 for a human to triage before Phase 2 (Docker/Fargate) work begins.
